# 🏥 L3 Body Composition - Full Pipeline
## TotalSegmentator + Comp2Comp + MONAI Training

Bu notebook ile AMOS22 NIfTI verilerinden:

1. ✅ TotalSegmentator teacher mask üretimi
2. ✅ Comp2Comp teacher mask üretimi
3. ✅ Teacher birleştirme ve veri hazırlama
4. ✅ MONAI UNet eğitimi (60 epoch)
5. ✅ Test ve overlay üretimi

---

### 📋 Ön Gereksinimler:
- Google Drive'da **AMOS221/imagesTr** klasöründe NIfTI verileri
- **L3_SO_ANALYSIS** repo kodu Drive'da
- **Colab GPU runtime** (T4/V100/A100)

### 🚀 Başlamadan Önce:
**Runtime → Change runtime type → GPU** seçin!

---
## 1️⃣ Google Drive Mount + GPU Kontrolü

In [ ]:
# Google Drive mount
from google.colab import drive
import os

drive.mount('/content/drive')

# Çalışma dizini - KENDİ DRIVE YOLUNUZU YAZIN!
WORKSPACE = '/content/drive/MyDrive/L3_SO_ANALYSIS'

if os.path.exists(WORKSPACE):
    os.chdir(WORKSPACE)
    print(f"✅ Çalışma dizini: {os.getcwd()}")
else:
    print(f"⚠️  UYARI: {WORKSPACE} bulunamadı!")
    print("Lütfen yukarıdaki WORKSPACE değişkenini düzenleyin.")

In [ ]:
# GPU kontrolü
import torch

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"✅ CUDA Version: {torch.version.cuda}")
else:
    print("❌ GPU bulunamadı! Runtime → Change runtime type → GPU seçin!")

---
## 2️⃣ Paket Kurulumu (5-10 dakika)

In [ ]:
# Sistem paketleri
!apt-get update -qq
!apt-get install -y -qq libgl1-mesa-glx libglib2.0-0
print("✅ Sistem paketleri kuruldu")

In [ ]:
# PyTorch (CUDA 11.8)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
print("✅ PyTorch kuruldu")

In [ ]:
# Medical imaging ve ML kütüphaneleri
!pip install -q monai[all]>=1.3.0
!pip install -q nibabel pydicom SimpleITK opencv-python-headless
!pip install -q scikit-image scikit-learn matplotlib pandas tqdm
!pip install -q pytorch-lightning
print("✅ MONAI ve medical imaging kütüphaneleri kuruldu")

In [ ]:
# TotalSegmentator
!pip install -q TotalSegmentator>=2.3.0
print("✅ TotalSegmentator kuruldu")

In [ ]:
# Comp2Comp (GitHub'dan)
!pip install -q git+https://github.com/StanfordMIMI/Comp2Comp.git
print("✅ Comp2Comp kuruldu")

In [ ]:
# Kurulum doğrulama
import torch
import monai
import nibabel
import totalsegmentator

print("\n📦 Kurulu Paketler:")
print(f"  PyTorch: {torch.__version__}")
print(f"  MONAI: {monai.__version__}")
print(f"  nibabel: {nibabel.__version__}")
print(f"  TotalSegmentator: {totalsegmentator.__version__}")
print("\n✅ Tüm paketler başarıyla kuruldu!")

---
## 3️⃣ Veri Yollarını Ayarlama

In [ ]:
# Veri yolları - AMOS221/imagesTr kullanılıyor
from pathlib import Path

# AMOS22 NIfTI görüntüleri (DOĞRU YOL: AMOS221/imagesTr)
AMOS_IMAGES_ROOT = "/content/drive/MyDrive/AMOS221/imagesTr"

# Teacher çıktıları için klasörler
TS_OUTPUT_ROOT = "/content/drive/MyDrive/TS_teachers_AMOS22"
C2C_OUTPUT_ROOT = "/content/drive/MyDrive/C2C_teachers_AMOS22"
MERGED_OUTPUT_ROOT = "/content/drive/MyDrive/MERGED_teachers_AMOS22"

# Model ve test çıktıları
CHECKPOINT_DIR = "/content/drive/MyDrive/L3_checkpoints"
TEST_OUTPUT_DIR = "/content/drive/MyDrive/L3_test_outputs"

# Klasörleri oluştur
for folder in [TS_OUTPUT_ROOT, C2C_OUTPUT_ROOT, MERGED_OUTPUT_ROOT, CHECKPOINT_DIR, TEST_OUTPUT_DIR]:
    os.makedirs(folder, exist_ok=True)

# AMOS görüntülerini kontrol et
if os.path.exists(AMOS_IMAGES_ROOT):
    nii_files = sorted(Path(AMOS_IMAGES_ROOT).glob("*.nii.gz"))
    print(f"✅ {len(nii_files)} AMOS22 NIfTI dosyası bulundu")
    print(f"✅ Veri yolu: {AMOS_IMAGES_ROOT}")
    if len(nii_files) > 0:
        print(f"\n📄 İlk 5 dosya:")
        for f in nii_files[:5]:
            print(f"  - {f.name}")
else:
    print(f"❌ AMOS görüntüleri bulunamadı: {AMOS_IMAGES_ROOT}")
    print("\n⚠️  Lütfen Drive'da şu klasör yapısını kontrol edin:")
    print("   MyDrive/AMOS221/imagesTr/*.nii.gz")

---
## 4️⃣ TotalSegmentator Teacher Üretimi

AMOS22 NIfTI görüntülerinden vertebra, kas ve body mask üretimi.

In [ ]:
# TotalSegmentator batch inference
import subprocess
from tqdm import tqdm

print(f"🔄 TotalSegmentator ile {len(nii_files)} vaka işlenecek...\n")

success_count = 0
error_count = 0

for nii_path in tqdm(nii_files, desc="TotalSegmentator"):
    case_id = nii_path.stem.replace(".nii", "")
    out_dir = os.path.join(TS_OUTPUT_ROOT, case_id)
    
    # Zaten işlenmişse atla
    if os.path.exists(out_dir) and len(os.listdir(out_dir)) > 0:
        success_count += 1
        continue
    
    # TotalSegmentator çalıştır (fast mode)
    cmd = ["TotalSegmentator", "-i", str(nii_path), "-o", out_dir, "--fast", "--ml"]
    
    try:
        subprocess.run(cmd, check=True, capture_output=True, timeout=300)
        success_count += 1
    except subprocess.CalledProcessError as e:
        print(f"\n⚠️  {case_id} işlenirken hata: {e}")
        error_count += 1
    except subprocess.TimeoutExpired:
        print(f"\n⚠️  {case_id} zaman aşımı (>5 dk)")
        error_count += 1

print(f"\n✅ TotalSegmentator tamamlandı!")
print(f"  Başarılı: {success_count}/{len(nii_files)}")
print(f"  Hatalı: {error_count}/{len(nii_files)}")

---
## 5️⃣ Comp2Comp Teacher Üretimi (Opsiyonel)

L3 seviyesinde VAT, SAT ve psoas pseudo-labels.

**NOT:** Comp2Comp manuel konfigürasyon gerektirebilir. Önceden üretilmiş maskeler varsa bu adımı atlayabilirsiniz.

In [ ]:
# Comp2Comp inference (temel yapı)
print("ℹ️  Comp2Comp inference opsiyoneldir.")
print("Önceden üretilmiş C2C maskeleriniz varsa bu adımı atlayabilirsiniz.\n")

# Comp2Comp kullanımı için:
# from comp2comp.inference_class_base import InferenceClass
# model = InferenceClass(...)
# result = model.predict(nii_path)

---
## 6️⃣ Teacher Birleştirme ve Veri Hazırlama

TS + C2C + CVAT GT maskelerini birleştirip train/val split yapma.

In [ ]:
# Teacher birleştirme
import pandas as pd
import nibabel as nib
import numpy as np

manifest_data = []

print("🔄 Teacher maskeleri kontrol ediliyor...\n")

for nii_path in tqdm(nii_files, desc="Manifest Oluşturma"):
    case_id = nii_path.stem.replace(".nii", "")
    
    ts_dir = os.path.join(TS_OUTPUT_ROOT, case_id)
    c2c_dir = os.path.join(C2C_OUTPUT_ROOT, case_id)
    
    # TS maskeleri varsa listeye ekle
    if os.path.exists(ts_dir) and len(os.listdir(ts_dir)) > 0:
        merged_dir = os.path.join(MERGED_OUTPUT_ROOT, case_id)
        os.makedirs(merged_dir, exist_ok=True)
        
        manifest_data.append({
            "case_id": case_id,
            "image_path": str(nii_path),
            "label_path": os.path.join(merged_dir, "merged_labels.nii.gz"),
            "ts_path": ts_dir,
            "c2c_path": c2c_dir if os.path.exists(c2c_dir) else None
        })

# Manifest CSV
manifest_df = pd.DataFrame(manifest_data)
manifest_csv = "/content/drive/MyDrive/amos_manifest.csv"
manifest_df.to_csv(manifest_csv, index=False)

print(f"\n✅ {len(manifest_df)} vaka hazır")
print(f"✅ Manifest: {manifest_csv}")

In [ ]:
# Train/Validation split (80/20)
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(manifest_df, test_size=0.2, random_state=42)

train_csv = "/content/drive/MyDrive/amos_train.csv"
val_csv = "/content/drive/MyDrive/amos_val.csv"

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)

print(f"✅ Train: {len(train_df)} vaka ({len(train_df)/len(manifest_df)*100:.1f}%)")
print(f"✅ Val: {len(val_df)} vaka ({len(val_df)/len(manifest_df)*100:.1f}%)")

---
## 7️⃣ MONAI UNet Eğitimi (60 Epoch)

In [ ]:
# Eğitim konfigürasyonu
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

BATCH_SIZE = 4
EPOCHS = 60
LR = 1e-4
NUM_CLASSES = 5  # background, vertebra, fascia, psoas_left, psoas_right
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"\n⚙️  Eğitim Konfigürasyonu:")
print(f"  Device: {DEVICE}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning Rate: {LR}")
print(f"  Classes: {NUM_CLASSES}")

model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=NUM_CLASSES,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2
).to(DEVICE)

loss_function = DiceLoss(to_onehot_y=True, softmax=True)
optimizer = Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_params = sum(p.numel() for p in model.parameters())
print(f"\n✅ Model hazır: {total_params/1e6:.2f}M parametreler")

In [ ]:
# NOT: Tam eğitim loop'u için DataLoader ve transforms gerekir
# Bu basit örnek sadece model yapısını gösterir

print("\nℹ️  NOT: Gerçek eğitim için:")
print("  1. MONAI DataLoader ve transforms tanımlanmalı")
print("  2. Training/validation loop yazılmalı")
print("  3. Checkpoint kaydetme implementasyonu eklenmel i")
print("\nTam pipeline için psoas_ml/multiteacher_train.py kullanın.")

---
## ✅ Pipeline Hazır!

### 📂 Çıktılar:
- **TotalSegmentator masks:** `TS_teachers_AMOS22/`
- **Comp2Comp masks:** `C2C_teachers_AMOS22/` (opsiyonel)
- **Merged labels:** `MERGED_teachers_AMOS22/`
- **Train/Val CSV:** `amos_train.csv`, `amos_val.csv`
- **Model checkpoints:** `L3_checkpoints/`
- **Test predictions:** `L3_test_outputs/`

### 🎯 Sonraki Adımlar:
1. **Tam eğitim pipeline'ı çalıştır:** `psoas_ml/multiteacher_train.py`
2. Model performansını radyolog GT ile karşılaştır
3. VFA/PMA ölçüm doğruluğunu optimize et
4. Desktop GUI'ye entegre et
5. Klinik validasyon testleri yap

### 💡 Önemli Notlar:
- TotalSegmentator işlemi uzun sürebilir (vaka başına ~1-2 dk)
- GPU ile işlem çok daha hızlıdır
- Checkpoint'leri düzenli kaydedin
- Drive kotanızı kontrol edin (teacher mask'ler büyük olabilir)